# Notebook 7: Prediction Pipeline

**Purpose:** This notebook allows you to run the trained AtrionNet model on **any single ECG record**
and see its P-wave predictions visualised directly in the notebook.

This is designed for:
- Live demonstrations during your thesis viva
- Verifying the model works on new records
- Generating visual evidence of P-wave detection

**Input:** A record index from the LUDB dataset

**Output:** A plot showing the ECG signal with detected P-waves highlighted

## Step 1: Setup

In [ ]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = str(Path(os.getcwd()).parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## Step 2: Load Model and Dataset

In [ ]:
from src.modeling.atrion_net import AtrionNetHybrid
from src.data_pipeline.ludb_loader import LUDBLoader

# Load model weights
MODEL_PATH = os.path.join(PROJECT_ROOT, 'weights', 'atrion_hybrid_best.pth')
model = AtrionNetHybrid(in_channels=12).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print(f"Model loaded from: {MODEL_PATH}")

# Load all records
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw', 'ludb')
loader = LUDBLoader(DATA_DIR)
signals, annotations = loader.get_all_data()
print(f"Dataset loaded: {len(signals)} records available.")

## Step 3: Run Prediction on a Single Record

Change `RECORD_INDEX` to any number between 0 and the total number of records
to run predictions on a different ECG.

In [ ]:
from src.engine.atrion_evaluator import get_instances_from_heatmap

# --- CHANGE THIS to inspect a different record ---
RECORD_INDEX = 0

# Normalize the signal for model input
sig_raw = signals[RECORD_INDEX].copy()
mu      = sig_raw.mean(axis=1, keepdims=True)
sigma   = sig_raw.std(axis=1, keepdims=True) + 1e-6
sig_norm = (sig_raw - mu) / sigma

# Run model inference
sig_tensor = torch.FloatTensor(sig_norm).unsqueeze(0).to(DEVICE)  # [1, 12, 5000]
with torch.no_grad():
    output = model(sig_tensor)

# Extract predicted P-wave instances from the heatmap
pred_heatmap = output['heatmap'][0].cpu().numpy()  # [1, 5000]
pred_width   = output['width'][0].cpu().numpy()    # [1, 5000]
pred_instances = get_instances_from_heatmap(pred_heatmap, pred_width, threshold=0.3)

print(f"Record {RECORD_INDEX}: {len(pred_instances)} P-wave instances detected.")
for idx, inst in enumerate(pred_instances):
    print(f"  Instance {idx+1}: Center={inst['center']}, Span={inst['span']}, Confidence={inst['confidence']:.3f}")

## Step 4: Visualise Predictions vs Ground Truth

This plot overlays the model's predicted P-wave regions (blue) against the ground truth (red)
on the Lead II ECG signal.

In [ ]:
lead_ii   = sig_raw[1]     # Lead II (raw, unnormalised for visibility)
time_axis = np.arange(len(lead_ii)) / 500.0
gt_waves  = annotations[RECORD_INDEX]['p_waves']

fig, axes = plt.subplots(3, 1, figsize=(16, 10))

# --- Top: ECG with GT and Predictions ---
axes[0].plot(time_axis, lead_ii, color='steelblue', linewidth=0.8, label='Lead II ECG')
for onset, peak, offset in gt_waves:
    axes[0].axvspan(onset/500, offset/500, color='red', alpha=0.25, label='Ground Truth P-wave')
for inst in pred_instances:
    s, e = inst['span']
    axes[0].axvspan(s/500, e/500, color='limegreen', alpha=0.30, label='Predicted P-wave')
axes[0].set_title(f'Record {RECORD_INDEX} — ECG Lead II with P-Wave Detection')
axes[0].set_ylabel('Amplitude (mV)')
handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axes[0].legend(by_label.values(), by_label.keys(), loc='upper right')

# --- Middle: Predicted Heatmap ---
hm = pred_heatmap.squeeze()
axes[1].plot(time_axis, hm, color='crimson', linewidth=1.2)
axes[1].axhline(0.3, color='gray', linestyle='--', label='Detection threshold (0.3)')
axes[1].set_title('Model Heatmap — P-Wave Center Probability')
axes[1].set_ylabel('Score')
axes[1].legend(loc='upper right')
axes[1].set_ylim(0, 1.05)

# --- Bottom: Predicted Mask ---
mask = output['mask'][0, 0].cpu().numpy()
axes[2].fill_between(time_axis, mask, color='orange', alpha=0.5, label='Predicted Mask')
axes[2].set_title('Model Mask — P-Wave Region Segmentation')
axes[2].set_ylabel('Score')
axes[2].set_xlabel('Time (seconds)')
axes[2].legend(loc='upper right')

plt.tight_layout()

pred_plot_path = os.path.join(PROJECT_ROOT, 'reports', 'plots', f'record_{RECORD_INDEX}_prediction.png')
plt.savefig(pred_plot_path, dpi=300)
plt.show()

print(f"Prediction plot saved to: {pred_plot_path}")

---
**Prediction pipeline complete. Proceed to `08_visualization/plots_and_graphs.ipynb`.**